In [52]:
import pandas as pd 
import numpy as np
import os 
from datetime import datetime, timedelta, date
from matplotlib import pyplot as plt
from dateutil.relativedelta import relativedelta
from scipy.stats import ttest_1samp
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None  # default='warn'

mypath = "C:\\Users\\erict\\OneDrive - Nanyang Technological University\\Documents\\Academia\\Papers - Completed\\\Greenwashing\\Code\\data2026"
os.chdir(mypath)

In [53]:
# Mapping Tables
# Map for permid <-> permid2 (to be deleted)
# Map for permid2 <-> RIC
# Map for industries to industry types

permids_map = pd.read_csv(mypath +"\\map_permid.csv", index_col=0)
permids_map = permids_map[permids_map["country"]=="United States"]
permids_map = permids_map[["permid", "permid2", "industry_type"]]

df_ref = pd.read_csv(mypath + "\\reference-data.csv")
df_ref = df_ref[df_ref["country"]=="USA"]
df_ref = df_ref.join(permids_map.set_index("permid"), on = "permid")
df_ref = df_ref[["permid", "permid2", "RIC", "industry_type" ]]
print(df_ref.head(5)) 

#industries = ["Consumer Cyclicals", "Consumer Non-Cyclicals", "Basic Materials", "Energy", "Utilities", "Technology", "Financials", "Healthcare"] 
#industry_map = {"others": ['999', 'Real Estate', 'Academic and Educational Services', "Government Activity"], "consumer" : ["Consumer Cyclicals", "Consumer Non-Cyclicals"], "brown": ["Basic Materials", "Energy", "Utilities", "Industrials"], "green" : ["Technology", "Financials", "Healthcare"] }
#industry_map = { v: k for k, l in industry_map.items() for v in l}
#print(industry_map) 

       permid                 permid2   RIC industry_type
0  4295899517  ScJQLAg8ZwTbARXbFpcHSC  SOMX         Green
1  4295899523  VaFvLPUUwbG28xyguZrS5G   NSH         Brown
2  4295899186  J3zhZQisLj9fBzmbu9CbiV  SHOR         Green
3  4295899194  3Xy4TzSQkbHQFYNkXJe2C3  IMUC         Green
4  4295899514  jVVeCWiD9mvLWhLn263kof   ZZZ         Brown


### Fama French Research Data And Indivudal Fundamental Factors

Downloaded from the Fama French website. Explicitly for the market factor as one of the regressor


In [54]:
df_mkt = pd.read_csv(mypath + "\\FFactors.csv")
df_mkt["Yr Month"] = df_mkt["Date"].map(lambda x : date(int(str(x)[0:4]), int(str(x)[4:6]), 1))
df_mkt["Mkt_RF"] = df_mkt["Mkt_RF"]*0.01
df_mkt = df_mkt[["Yr Month", "Mkt_RF"]]
df_mkt.to_csv(mypath+ "\\df_mkt.csv")
df_mkt.head()

,Yr Month,Mkt_RF
0,2004-12-01,0.0342
1,2005-01-01,-0.0275
2,2005-02-01,0.0188
3,2005-03-01,-0.0194
4,2005-04-01,-0.0261


### Firm Size Data (Market Capitalisation)

Computing market capitalisation for firm with share price and number of shares.

In [55]:
# 3. Read in prices for permids from raw crsp files 
# this takes a while to run and has been abstracted into df_price
df_price_raw = pd.read_csv(mypath + "\\raw_crsp_US_prices.csv", index_col=0)
df_price = df_price_raw.join(df_ref.set_index("RIC"), on="tic")  # permid inside df_ref
df_price = df_price.rename(columns = {"prccd": "Close", "cshoc": "NoOfShares"})
df_price["Yr Month"] = df_price["datadate"].map(lambda x : date(int(str(x)[0:4]), int(str(x)[4:6]), 1))
df_price = df_price[["permid2", "Yr Month", "Close", "NoOfShares"]]
df_price = df_price.groupby(["permid2", "Yr Month"], as_index=False).mean()

df_price.to_csv(mypath + "\\df_price.csv")
#print(df_price.head())

C:\Users\erict\AppData\Local\Temp\ipykernel_78256\1676268501.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_price_raw = pd.read_csv(mypath + "\\raw_crsp_US_prices.csv", index_col=0)


In [56]:
# obtaining mkt cap
df_price = pd.read_csv(mypath + "\\df_price.csv")
df_price["mkt_cap"] = np.log10((df_price["Close"] * df_price["NoOfShares"]).replace(0, np.nan))
df_mktcap = df_price[["permid2", "Yr Month", "mkt_cap"]]
df_mktcap.to_csv(mypath + "\\df_mktcap.csv")
print(df_mktcap.head(5))

# obtaining returns
df_price = df_price.sort_values(['permid2', 'Yr Month']).reset_index(drop=True)
df_price["ret"] = df_price.groupby('permid2')['Close'].pct_change()
df_ret = df_price[["permid2", "Yr Month", "ret", 'Close']]
df_ret.to_csv(mypath + "\\df_ret.csv")
print(df_ret.head(5))

                  permid2    Yr Month   mkt_cap
0  226xZbVHNSKmCP4R5s7fMT  2014-10-01  9.694716
1  226xZbVHNSKmCP4R5s7fMT  2014-11-01  9.719621
2  226xZbVHNSKmCP4R5s7fMT  2014-12-01  9.753601
3  226xZbVHNSKmCP4R5s7fMT  2015-01-01  9.760065
4  226xZbVHNSKmCP4R5s7fMT  2015-02-01  9.782870
                  permid2    Yr Month       ret      Close
0  226xZbVHNSKmCP4R5s7fMT  2014-10-01       NaN  29.681000
1  226xZbVHNSKmCP4R5s7fMT  2014-11-01  0.055150  31.317895
2  226xZbVHNSKmCP4R5s7fMT  2014-12-01  0.080329  33.833636
3  226xZbVHNSKmCP4R5s7fMT  2015-01-01  0.013518  34.291000
4  226xZbVHNSKmCP4R5s7fMT  2015-02-01  0.052110  36.077895


### LSEG Asset4 Data

THis data is used to retrieve company-specific scores on an objective and observable scale performance. Only United States firms' data are filtered for analysis.

In [57]:
df_asset4 = pd.read_csv(mypath + "\\df_asset4_raw.csv", index_col=0) 
esg_scores = ['ESG Score', "Environment Pillar Score", "Social Pillar Score", "Governance Pillar Score"]
df_asset4["Yr Month"] = df_asset4["Date"].map(lambda x : date(int(str(x)[:4]), int(str(x)[5:7]) ,1) )
df_asset4["Year"] = df_asset4["Date"].map(lambda x : int(str(x)[:4]))


In [58]:
df_asset4 = df_asset4[df_asset4["country"]=="United States"]
df_asset4 = df_asset4.join(df_ref.set_index("permid"), on = "permid")
df_asset4 = df_asset4.drop("permid",axis=1)
df_asset4 = df_asset4.drop("industry", axis = 1)

df_asset4 = df_asset4[["permid2", "industry_type", "Year", 'ESG Score', "Environment Pillar Score", "Social Pillar Score", "Governance Pillar Score"]]
df_asset4grp = df_asset4.groupby(["permid2", "Year"])[esg_scores].mean().reset_index()
print(df_asset4grp.head())
df_asset4.to_csv(mypath + "\\df_asset4.csv")

                  permid2  Year  ESG Score  Environment Pillar Score  \
0  222HWwE2amGnhKR4sWFpkY  2019  17.953156                  0.000000   
1  222HWwE2amGnhKR4sWFpkY  2020  28.651754                  0.000000   
2  222HWwE2amGnhKR4sWFpkY  2021  25.095378                  2.932961   
3  226xZbVHNSKmCP4R5s7fMT  2015  64.704567                 63.385854   
4  226xZbVHNSKmCP4R5s7fMT  2016  73.640853                 75.855468   

   Social Pillar Score  Governance Pillar Score  
0            27.710404                29.737533  
1            45.051658                46.681113  
2            40.601079                36.393400  
3            87.695019                36.143984  
4            94.281875                43.920640  


### ESG sentiment metrics from MarketPschy

In [59]:
##  Actual data input -> ESG RNA scores
df_esg = pd.read_csv("df_esg.csv")
df_esg = df_esg[df_esg["permid2"].isin(df_asset4["permid2"].unique())]
df_esg = df_esg[["permid2", "Yr Month", "ESG", "EnvironmentalPillar", "GovernancePillar", "SocialPillar" ]]
df_esg.to_csv(mypath + "\\df_esg.csv")

### Firm Momentum data
The data here is to calculate the monthly returns and then the momentum. First calculate the firms' monthly returns.  

In [60]:
# Obtain monthly returns and momentum
df_ret = pd.read_csv(mypath + "\\df_ret.csv")
df_mom = df_ret[["permid2", "Yr Month", "ret"]]
#df_mom = df_mom.sort_values(['permid2', 'Yr Month']).reset_index(drop=True)

# Create lagged Close column, grouped by permid2
df_mom['Close_lag1'] = df_mom.groupby('permid2')['ret'].shift(1)
df_mom['Close_lag2'] = df_mom.groupby('permid2')['ret'].shift(2)
df_mom['Close_lag3'] = df_mom.groupby('permid2')['ret'].shift(3)
df_mom['Close_lag4'] = df_mom.groupby('permid2')['ret'].shift(4)
df_mom['Close_lag5'] = df_mom.groupby('permid2')['ret'].shift(5)
df_mom['Close_lag6'] = df_mom.groupby('permid2')['ret'].shift(6)
df_mom['Close_lag7'] = df_mom.groupby('permid2')['ret'].shift(7)
df_mom['Close_lag8'] = df_mom.groupby('permid2')['ret'].shift(8)
df_mom['Close_lag9'] = df_mom.groupby('permid2')['ret'].shift(9)
df_mom['Close_lag10'] = df_mom.groupby('permid2')['ret'].shift(10)
df_mom['Close_lag11'] = df_mom.groupby('permid2')['ret'].shift(11)
df_mom['Close_lag12'] = df_mom.groupby('permid2')['ret'].shift(12)

lag_cols = [f"Close_lag{i}" for i in range(1, 7)]
df_mom["MomentumL6m"] = df_mom[lag_cols].mean(axis=1)

df_mom = df_mom[["permid2", "Yr Month", "ret", "MomentumL6m" ]]
df_mom.to_csv(mypath + "\\df_mom.csv")
print(df_mom.head())

                  permid2    Yr Month       ret  MomentumL6m
0  226xZbVHNSKmCP4R5s7fMT  2014-10-01       NaN          NaN
1  226xZbVHNSKmCP4R5s7fMT  2014-11-01  0.055150          NaN
2  226xZbVHNSKmCP4R5s7fMT  2014-12-01  0.080329     0.055150
3  226xZbVHNSKmCP4R5s7fMT  2015-01-01  0.013518     0.067739
4  226xZbVHNSKmCP4R5s7fMT  2015-02-01  0.052110     0.049666


### Firm Book Market data
The data here is to retrieve the firms' fundamentals in particular book to market value (bm) from wrds.

In [61]:
# 4. Obtain the firm fundamental data from WRDS
df_fundamentals = pd.read_csv(mypath + "\\raw_crsp_US_fun_ratios.csv")
df_fundamentals["Yr Month"] = df_fundamentals["public_date"].map(lambda x : date(int(str(x)[0:4]), int(str(x)[4:6]), 1))
df_fundamentals = df_fundamentals.join(df_ref.set_index("RIC"), on="TICKER") 
df_bm = df_fundamentals[["permid2", "Yr Month", 'bm']]

print(df_bm.head())
df_bm.to_csv(mypath + "\\df_bm.csv")

                  permid2    Yr Month     bm
0  imqPbj4yARj8V3isZ9iWpo  1997-12-01  0.499
1  imqPbj4yARj8V3isZ9iWpo  1998-01-01  0.499
2  imqPbj4yARj8V3isZ9iWpo  1998-02-01  0.499
3  imqPbj4yARj8V3isZ9iWpo  1998-03-01  0.480
4  imqPbj4yARj8V3isZ9iWpo  1998-04-01  0.480


### Do Ordinary Least Square Regression to obtain betas for the firms

In [62]:
# firms'  fundamental data
df_bm = pd.read_csv(mypath + "\\df_bm.csv", index_col=0) # contain firm fundamental data
df_mom = pd.read_csv(mypath + "\\df_mom.csv", index_col=0)
df_mktcap = pd.read_csv(mypath + "\\df_mktcap.csv", index_col=0)
df_mkt = pd.read_csv(mypath + "\\df_mkt.csv", index_col=0)
print(df_mkt.head())

df_fundamentals = df_bm.merge(df_mom, on = ["permid2", "Yr Month"])
df_fundamentals = df_fundamentals.merge(df_mktcap, on = ["permid2", "Yr Month"])

df_fundamentals = df_fundamentals.merge(df_mkt, on = ["Yr Month"])
print(df_fundamentals.head())
print(df_fundamentals.columns)
df_fundamentals = df_fundamentals[["permid2", "Yr Month", "bm", "MomentumL6m", "mkt_cap", "Mkt_RF"]]
df_fundamentals.to_csv(mypath + "\\df_fundamentals.csv")
print(df_fundamentals.head())

df_ret = pd.read_csv(mypath + "\\df_ret.csv", index_col=0)

     Yr Month  Mkt_RF
0  2004-12-01  0.0342
1  2005-01-01 -0.0275
2  2005-02-01  0.0188
3  2005-03-01 -0.0194
4  2005-04-01 -0.0261
                  permid2    Yr Month     bm       ret  MomentumL6m   mkt_cap  \
0  imqPbj4yARj8V3isZ9iWpo  2004-12-01  0.544  0.184682     0.005710  8.038572   
1  imqPbj4yARj8V3isZ9iWpo  2005-01-01  0.544  0.322396     0.032170  8.159934   
2  imqPbj4yARj8V3isZ9iWpo  2005-02-01  0.544  0.160146     0.086753  8.226097   
3  imqPbj4yARj8V3isZ9iWpo  2005-03-01  0.310 -0.002954     0.123279  8.225976   
4  imqPbj4yARj8V3isZ9iWpo  2005-04-01  0.310 -0.044243     0.116007  8.209411   

   Mkt_RF  
0  0.0342  
1 -0.0275  
2  0.0188  
3 -0.0194  
4 -0.0261  
Index(['permid2', 'Yr Month', 'bm', 'ret', 'MomentumL6m', 'mkt_cap', 'Mkt_RF'], dtype='object')
                  permid2    Yr Month     bm  MomentumL6m   mkt_cap  Mkt_RF
0  imqPbj4yARj8V3isZ9iWpo  2004-12-01  0.544     0.005710  8.038572  0.0342
1  imqPbj4yARj8V3isZ9iWpo  2005-01-01  0.544     0.032170  8.

In [63]:
df_reg = df_bm.merge(df_mom, on =["permid2","Yr Month"], how="left")
df_reg = df_reg.merge(df_mktcap, on =["permid2","Yr Month"], how="left")
df_reg = df_reg[["permid2", "Yr Month", "ret", "bm", "MomentumL6m", "mkt_cap"  ]]
df_reg = df_reg.merge(df_mkt, on =["Yr Month"], how="left")
df_reg = df_reg[["permid2", "Yr Month", "ret", "bm", "MomentumL6m", "mkt_cap" , "Mkt_RF" ]]
# filter off data before 2005-01-01
df_reg['Yr Month'] = pd.to_datetime(df_reg['Yr Month'])  # convert once
df_reg = df_reg[df_reg['Yr Month'] >= '2005-01-01']
print(df_reg.head())
df_reg.columns

                   permid2   Yr Month       ret     bm  MomentumL6m   mkt_cap  \
85  imqPbj4yARj8V3isZ9iWpo 2005-01-01  0.322396  0.544     0.032170  8.159934   
86  imqPbj4yARj8V3isZ9iWpo 2005-02-01  0.160146  0.544     0.086753  8.226097   
87  imqPbj4yARj8V3isZ9iWpo 2005-03-01 -0.002954  0.310     0.123279  8.225976   
88  imqPbj4yARj8V3isZ9iWpo 2005-04-01 -0.044243  0.310     0.116007  8.209411   
89  imqPbj4yARj8V3isZ9iWpo 2005-05-01 -0.162475  0.310     0.102032  8.132409   

    Mkt_RF  
85 -0.0275  
86  0.0188  
87 -0.0194  
88 -0.0261  
89  0.0365  


Index(['permid2', 'Yr Month', 'ret', 'bm', 'MomentumL6m', 'mkt_cap', 'Mkt_RF'], dtype='object')

In [64]:
def hac_cov_matrix(X, resid, n, k, maxlags):
    """
    Newey-West HAC covariance matrix for OLS coefficients.
    X: (n, k) design matrix
    resid: (n,) residuals
    maxlags: truncation lag (bandwidth)
    """
    XtX_inv = np.linalg.inv(X.T @ X)
    
    # Score (moment) contributions: X_i * resid_i, shape (n, k)
    scores = X * resid[:, None]
    
    # S_0: contemporaneous term
    S = scores.T @ scores
    
    # Add weighted autocovariance terms
    for lag in range(1, maxlags + 1):
        weight = 1 - lag / (maxlags + 1)  # Bartlett kernel
        Gamma = scores[lag:].T @ scores[:-lag]
        S += weight * (Gamma + Gamma.T)
    
    # Sandwich: (X'X)^-1 * S * (X'X)^-1 * n  (small-sample style; see note below)
    meat = S
    cov = XtX_inv @ meat @ XtX_inv
    return cov


def ols_per_group_hac(df, y_col, x_cols, group_col='permid2', intercept=True, min_obs=24, maxlags=None):
    sub = df[[group_col, y_col] + x_cols].dropna()

    k = len(x_cols) + (1 if intercept else 0)
    coef_names = (['const'] if intercept else []) + x_cols
    rows = []

    for permid, g in sub.groupby(group_col, sort=False):
        n = len(g)
        if n < min_obs or n <= k:
            continue

        g = g.sort_index() if 'Yr Month' not in g.columns else g.sort_values('Yr Month')

        if intercept:
            X = np.column_stack([np.ones(n), g[x_cols].values])
        else:
            X = g[x_cols].values  # already (n, k), no need for column_stack

        y = g[y_col].values

        beta, _, rank, _ = np.linalg.lstsq(X, y, rcond=None)
        if rank < k:
            continue

        fitted = X @ beta
        resid = y - fitted

        lags = maxlags if maxlags is not None else int(np.floor(4 * (n / 100) ** (2/9)))
        lags = max(lags, 1)

        cov_beta = hac_cov_matrix(X, resid, n, k, lags)
        se_beta = np.sqrt(np.diag(cov_beta))

        dof = n - k
        t_stats = beta / se_beta
        p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), dof))

        tss = ((y - y.mean()) ** 2).sum()
        rss = (resid ** 2).sum()
        r2 = 1 - rss / tss if tss > 0 else np.nan

        row = {'permid2': permid, 'n_obs': n, 'r_squared': r2, 'hac_lags': lags}
        for i, name in enumerate(coef_names):
            row[f'{name}_coef'] = beta[i]
            row[f'{name}_se'] = se_beta[i]
            row[f'{name}_t'] = t_stats[i]
            row[f'{name}_p'] = p_values[i]

        rows.append(row)

    out = pd.DataFrame(rows).set_index('permid2')
    return out

In [ ]:
results = ols_per_group_hac(df_reg, y_col='ret', x_cols=['Mkt_RF', 'bm', 'MomentumL6m', 'mkt_cap'], min_obs=24)
results_0 = ols_per_group_hac(df_reg, y_col='ret', x_cols=['Mkt_RF', 'bm', 'MomentumL6m', 'mkt_cap'], min_obs=24, intercept= False)

In [ ]:
res = ["permid2", "Mkt_RF_coef", 'bm_coef', 'MomentumL6m_coef', 'mkt_cap_coef', 'const_coef']
res_p = ["Mkt_RF_p", 'bm_p', 'MomentumL6m_p', 'mkt_cap_p', 'const_p']
results = results.reset_index() 
df_betas = results[res]
df_betas_p = results[res_p]
df_betas.to_csv(mypath + "\\df_betas.csv") 

res = ["permid2", "Mkt_RF_coef", 'bm_coef', 'MomentumL6m_coef', 'mkt_cap_coef']
res_p = ["Mkt_RF_p", 'bm_p', 'MomentumL6m_p', 'mkt_cap_p']
results_0 = results_0.reset_index() 
df_betas_0 = results_0[res]
df_betas_p = results_0[res_p]
df_betas_0.to_csv(mypath + "\\df_betas_0.csv") 